# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > 练习实验：协同过滤推荐系统

在本练习中，你将实现协同过滤来构建一个电影推荐系统。

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [ 1 - 符号说明](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐](#6)
- [ 7 - 恭喜！](#7)




_**注意：** 为了防止自动评分器出错，你不允许编辑或删除本实验中的非评分单元格。请也不要添加任何新单元格。
**当你通过本作业后**，如果想尝试任何非评分代码，可以按照本笔记本底部的说明操作。_

##  软件包 <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用大家熟悉的 NumPy 和 Tensorflow 软件包。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 符号说明


|General <br />  Notation  | Description| Python (if any) |
|:-------------|:------------------------------------------------------------||
| $r(i,j)$     | scalar; = 1  if user j rated movie i  = 0  otherwise             ||
| $y(i,j)$     | scalar; = rating given by user j on movie  i    (if r(i,j) = 1 is defined) ||
|$\mathbf{w}^{(j)}$ | vector; parameters for user j ||
|$b^{(j)}$     |  scalar; parameter for user j ||
| $\mathbf{x}^{(i)}$ |   vector; feature ratings for movie i        ||     
| $n_u$        | number of users |num_users|
| $n_m$        | number of movies | num_movies |
| $n$          | number of features | num_features                    |
| $\mathbf{X}$ |  matrix of vectors $\mathbf{x}^{(i)}$         | X |
| $\mathbf{W}$ |  matrix of vectors $\mathbf{w}^{(j)}$         | W |
| $\mathbf{b}$ |  vector of bias parameters $b^{(j)}$ | b |
| $\mathbf{R}$ | matrix of elements $r(i,j)$                    | R |



<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并将其应用于电影评分数据集。
协同过滤推荐系统的目标是生成两个向量：对于每个用户，一个体现用户电影品味的「参数向量」。对于每部电影，一个相同大小的特征向量，体现电影的某些描述。两个向量的点积加上偏置项应该产生用户对该电影评分的估计。

下面的图表详细说明了这些向量是如何学习的。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

现有的评分以矩阵形式提供，如图所示。$Y$ 包含评分；0.5 到 5（含），步长为 0.5。如果电影未被评分则为 0。$R$ 在电影已被评分的位置为 1。电影在行中，用户在列中。每个用户有一个参数向量 $w^{user}$ 和偏置。每部电影有一个特征向量 $x^{movie}$。这些向量通过使用现有的用户/电影评分作为训练数据同时学习。上面显示了一个训练示例：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须满足所有用户，而用户向量 $w^{user}$ 必须满足所有电影。这就是这种方法名称的来源——所有用户协作生成评分集。

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

一旦特征向量和参数被学习，它们就可以用来预测用户可能对未评分电影的评分。这在上面的图表中有所展示。该方程是为用户一预测电影零评分的一个示例。


在本练习中，你将实现函数 `cofiCostFunc`，该函数计算协同过滤
目标函数。实现目标函数后，你将使用 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细说明将在实验中使用的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
该数据集源自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。   
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集有 9000 部电影，由 600 个用户评分。数据集已缩小规模，专注于 2000 年以来的电影。该数据集包含 0.5 到 5 的评分，步长为 0.5。缩小后的数据集有 $n_u = 443$ 个用户，$n_m= 4778$ 部电影。

下面，你将把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（一个 $n_m \times n_u$ 矩阵）存储评分 $y^{(i,j)}$。矩阵 $R$ 是一个二值指示矩阵，其中 $R(i,j) = 1$ 表示用户 $j$ 对电影 $i$ 进行了评分，否则 $R(i,j)=0$。

在本练习的这一部分，你还将使用
矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$：

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$ 的第 $i$ 行对应于
第 $i$ 部电影的特征向量 $x^{(i)}$，而
$\mathbf{W}$ 的第 $j$ 行对应于第 $j$ 个用户的参数向量 $\mathbf{w}^{(j)}$。
$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维
向量。在本练习中，你将使用 $n=10$，
因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 有 10 个元素。
相应地，$\mathbf{X}$ 是一个
$n_m \times 10$ 矩阵，$\mathbf{W}$ 是一个 $n_u \times 10$ 矩阵。

我们将首先加载电影评分数据集以了解数据结构。
我们将使用电影数据集加载 $Y$ 和 $R$。
我们还将使用预计算的值加载 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。这些值将在实验后面学习，但我们将使用预计算的值来开发代价模型。

In [2]:
# 加载数据
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
# 从矩阵中，我们可以计算平均评分等统计信息。
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习
算法。你将从实现目标函数开始。

在电影推荐的场景中，协同过滤算法
考虑一组 $n$ 维参数向量
$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，其中
模型将电影 $i$ 对用户 $j$ 的评分预测为
$y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(j)}$。给定一个由
某些用户对某些电影产生的评分组成的数据集，你希望
学习参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},
\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，使得产生最佳拟合（最小化
平方误差）。

你将完成 cofiCostFunc 中的代码来计算协同过滤的代价
函数。


<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤代价函数由以下公式给出
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \left[ \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+ \underbrace{\left[
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
\right]}_{正则化}
\tag{1}$$
(1) 中的第一个求和是「对于所有 $r(i,j)$ 等于 $1$ 的 $i$、$j$」，可以写成：

$$
= \left[ \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+\text{正则化}
$$

你现在应该编写 cofiCostFunc（协同过滤代价函数）来返回这个代价。

<a name="ex01"></a>
### 练习 1

**For 循环实现：**  
首先使用 for 循环实现代价函数。
考虑分两步开发代价函数。首先，开发不含正则化的代价函数。下面提供了一个不包含正则化的测试用例来测试你的实现。一旦运行正常，再添加正则化并运行包含正则化的测试。注意，你只应在 $R(i,j) = 1$ 时才累加用户 $j$ 和电影 $i$ 的代价。

In [17]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w_j = W[j, :]
        b_j = b[0, j]
        for i in range(nm):
            y_ij = Y[i, j]
            x_i = X[i, :]
            r_ij = R[i,j]
            
            J += np.square(r_ij * (np.dot(w_j, x_i) + b_j - y_ij))
    
    J = J/2 + lambda_ / 2 * (np.sum(np.square(X)) + np.sum(np.square(W)))
    
    ### END CODE HERE ### 

    return J

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    你可以使用两个 for 循环来构建代码，类似于 (1) 中的求和。   
    首先实现不含正则化的代码。   
    注意 (1) 中的一些元素是向量。使用 np.dot()。你也可以使用 np.square()。
    仔细注意哪些元素由 i 索引，哪些由 j 索引。不要忘记除以二。
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
    这里有更多细节。下面的代码在使用前从矩阵中提取每个元素。
    也可以直接引用矩阵。
    此代码不包含正则化。
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>最后手段（完整非正则化实现）</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化只是对 W 数组和 X 数组的每个元素求平方，然后将所有平方元素求和。
     你可以使用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化细节</b></font></summary>
    
```python 
    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>

    


In [18]:
# 缩小数据集大小以加快运行速度
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# 评估代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**预期输出（lambda = 0）**：  
$13.67$。

In [19]:
# 使用正则化评估代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**预期输出**：

28.09

In [ ]:
# 公共测试
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

**向量化实现**

创建向量化实现来计算 $J$ 非常重要，因为它稍后在优化过程中会被多次调用。所使用的线性代数不是本系列的重点，因此提供了实现。如果你是线性代数专家，请随意在不参考下面代码的情况下创建你的版本。

运行下面的代码并验证它产生与非向量化版本相同的结果。

In [20]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [21]:
# 评估代价函数
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# 使用正则化评估代价函数
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**预期输出**：  
Cost: 13.67  
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

在完成协同过滤代价
函数的实现后，你可以开始训练算法来为自己制作
电影推荐。

在下面的单元格中，你可以输入自己的电影选择。算法将为你提供推荐！我们根据自己的偏好填写了一些值，但在你使用我们的选择运行正常后，你应该将其更改以匹配你的品味。
数据集中所有电影的列表在文件 [movie list](data/small_movie_list.csv) 中。

In [22]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          # 初始化我的评分

# 查看文件 small_movie_list.csv 获取数据集中每部电影的 ID
# 例如，Toy Story 3 (2010) 的 ID 为 2700，所以要将其评分设为 "5"，可以设置
my_ratings[2700] = 5 

# 或者假设你不喜欢 Persuasion (2007)，你可以设置
my_ratings[2609] = 2;

# 我们选择了一些喜欢/不喜欢的电影，我们给出的
# 评分如下：
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，让我们将这些评论添加到 $Y$ 和 $R$ 中，并对评分进行归一化。

In [23]:
# 重新加载评分
Y, R = load_ratings_small()

# 将新用户评分添加到 Y
Y = np.c_[my_ratings, Y]

# 将新用户指示矩阵添加到 R
R = np.c_[(my_ratings != 0).astype(int), R]

# 对数据集进行归一化
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择 Adam 优化器。

In [24]:
#  有用的值
num_movies, num_users = Y.shape
num_features = 100

# 设置初始参数 (W, X)，使用 tf.Variable 来跟踪这些变量
tf.random.set_seed(1234) # 以获得一致的结果
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# 实例化优化器
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在让我们训练协同过滤模型。这将学习参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。

同时学习 $w$、$b$ 和 $x$ 所涉及的操作不属于 TensorFlow 神经网络包中典型的「层」。因此，课程 2 中使用的流程：Model、Compile()、Fit()、Predict() 并不直接适用。相反，我们可以使用自定义训练循环。

回顾之前实验中梯度下降的步骤。
- 重复直到收敛：
    - 计算前向传播
    - 计算损失相对于参数的导数
    - 使用学习率和计算的导数更新参数
    
TensorFlow 具有为你计算导数的奇妙能力。这在下面展示。在 `tf.GradientTape()` 部分内，对 TensorFlow 变量的操作被跟踪。当稍后调用 `tape.gradient()` 时，它将返回损失相对于被跟踪变量的梯度。然后可以使用优化器将梯度应用于参数。
这是对 TensorFlow 和其他机器学习框架有用功能的非常简短的介绍。更多信息可以通过在感兴趣的框架内研究「自定义训练循环」来找到。
    


In [25]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # 使用 TensorFlow 的 GradientTape
    # 来记录计算代价所使用的操作
    with tf.GradientTape() as tape:

        # 计算代价（前向传播包含在代价中）
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # 使用梯度带自动检索
    # 可训练变量相对于损失的梯度
    grads = tape.gradient( cost_value, [X,W,b] )

    # 通过更新变量的值来运行一步梯度下降
    # 以最小化损失
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # 定期记录日志
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136168.7
Training loss at iteration 40: 51863.3
Training loss at iteration 60: 24598.8
Training loss at iteration 80: 13630.4
Training loss at iteration 100: 8487.6
Training loss at iteration 120: 5807.7
Training loss at iteration 140: 4311.6
Training loss at iteration 160: 3435.2
Training loss at iteration 180: 2902.1


<a name="6"></a>
## 6 - 推荐
下面，我们计算所有电影和用户的评分，并显示推荐的电影。这些基于上面作为 `my_ratings[]` 输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，你计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。这可以使用矩阵乘法对所有评分进行计算。

In [26]:
# 使用训练好的权重和偏置进行预测
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

# 恢复均值
pm = p + Ymean

my_predictions = pm[:,0]

# 对预测结果排序
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 4.49 for movie My Sassy Girl (Yeopgijeogin geunyeo) (2001)
Predicting rating 4.48 for movie Martin Lawrence Live: Runteldat (2002)
Predicting rating 4.48 for movie Memento (2000)
Predicting rating 4.47 for movie Delirium (2014)
Predicting rating 4.47 for movie Laggies (2014)
Predicting rating 4.47 for movie One I Love, The (2014)
Predicting rating 4.46 for movie Particle Fever (2013)
Predicting rating 4.45 for movie Eichmann (2007)
Predicting rating 4.45 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.45 for movie Into the Abyss (2011)


Original vs Predicted ratings:

Original 5.0, Predicted 4.90 for Shrek (2001)
Original 5.0, Predicted 4.84 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.13 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predicted 4.88 for Harry Potter and the Chamber of Secrets (2002)
Original 5.0, Predic

在实践中，可以利用额外信息来增强我们的预测。上面，前几百部电影的预测评分在一个小范围内。我们可以通过从那些排名靠前的电影中选择平均评分高且评分数量超过 20 的电影来增强上述内容。本节使用 [Pandas](https://pandas.pydata.org/) 数据框，它具有许多方便的排序功能。

In [27]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
1743,4.030965,4.252336,107,"Departed, The (2006)"
2112,3.985287,4.238255,149,"Dark Knight, The (2008)"
211,4.477792,4.122642,159,Memento (2000)
929,4.887053,4.118919,185,"Lord of the Rings: The Return of the King, The..."
2700,4.796530,4.109091,55,Toy Story 3 (2010)
653,4.357304,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
1122,4.004469,4.006494,77,Shaun of the Dead (2004)
1841,3.980647,4.000000,61,Hot Fuzz (2007)
3083,4.084633,3.993421,76,"Dark Knight Rises, The (2012)"
2804,4.434171,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...


<a name="7"></a>
## 7 - 恭喜！ <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个有用的推荐系统！

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想尝试任何非评分代码，请点击此处。</b></font></summary>
    <p><i><b>重要提示：请仅在你已通过作业后再执行此操作，以避免自动评分器出现问题。</b></i>
    <ol>
        <li> 在笔记本菜单中，点击 “View” > “Cell Toolbar” > “Edit Metadata”</li>
        <li> 点击你想要锁定/解锁的代码单元格旁边的 “Edit Metadata” 按钮</li>
        <li> 将 “editable” 的属性值设置为：
            <ul>
                <li> “true” 如果你想解锁它 </li>
                <li> “false” 如果你想锁定它 </li>
            </ul>
        </li>
        <li> 在笔记本菜单中，点击 “View” > “Cell Toolbar” > “None” </li>
    </ol>
    <p> 以下是上述步骤的简短演示：
        <br>
        <img src="https://drive.google.com/uc?export=view&id=14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>